# WF-001 Stage-1 Demo (Pure Python)

This notebook is a **pure-Python** step-by-step demonstration of the WF-001 stage-1 pipeline.

Instead of calling the CLI via subprocess, it:

- adds `src/` to `sys.path`
- loads the YAML config via `rpbench.config.load_config`
- calls `rpbench.runners.run_demo.run_demo(cfg)` directly
- writes outputs under the repo's `data/` directory

> The canonical path is still the CLI (`scripts/run_demo.py`), but this notebook shows the importable API path.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
os.chdir(REPO_ROOT)

SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = REPO_ROOT / 'data'
RUNS_DIR = DATA_DIR / 'outputs' / 'runs'
REPORT_DIR = REPO_ROOT / 'reports'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [2]:
from rpbench.config import load_config
from rpbench.runners.run_demo import run_demo
from rpbench.utils.io import save_jsonl

cfg = load_config(REPO_ROOT / 'configs' / 'demo' / 'autompg.yaml')
cfg.output_root = str(RUNS_DIR)
base_seed, trial_seeds = cfg.resolve_trial_seeds()
print('Seed batch:', {
    'mode': cfg.seed_batch.mode,
    'base_seed': base_seed,
    'count': len(trial_seeds),
    'expanded_seeds': trial_seeds,
})


records = run_demo(cfg)

out_path = Path(cfg.output_root) / 'demo_results.jsonl'
save_jsonl(records, out_path)

print(f'Saved {len(records)} records to {out_path}')


Seed batch: {'mode': 'fixed', 'base_seed': 0, 'count': 5, 'expanded_seeds': [2968811710, 3964924996, 3141116543, 2613022947, 1874364848]}
  [1/40] Mech_RP eps=0.5 trial=0 seed=2968811710
  [2/40] Mech_RP eps=0.5 trial=1 seed=3964924996
  [3/40] Mech_RP eps=0.5 trial=2 seed=3141116543
  [4/40] Mech_RP eps=0.5 trial=3 seed=2613022947
  [5/40] Mech_RP eps=0.5 trial=4 seed=1874364848
  [6/40] Mech_RP eps=1.0 trial=0 seed=2968811710
  [7/40] Mech_RP eps=1.0 trial=1 seed=3964924996
  [8/40] Mech_RP eps=1.0 trial=2 seed=3141116543
  [9/40] Mech_RP eps=1.0 trial=3 seed=2613022947
  [10/40] Mech_RP eps=1.0 trial=4 seed=1874364848
  [11/40] Mech_RP eps=2.0 trial=0 seed=2968811710
  [12/40] Mech_RP eps=2.0 trial=1 seed=3964924996
  [13/40] Mech_RP eps=2.0 trial=2 seed=3141116543
  [14/40] Mech_RP eps=2.0 trial=3 seed=2613022947
  [15/40] Mech_RP eps=2.0 trial=4 seed=1874364848
  [16/40] Mech_RP eps=4.0 trial=0 seed=2968811710
  [17/40] Mech_RP eps=4.0 trial=1 seed=3964924996
  [18/40] Mech_RP eps

In [3]:
from rpbench.reporting.tables import build_summary_table
from rpbench.reporting.figures import ols_plot_eps_vs_mse, plot_eps_vs_covariance_error
from rpbench.reporting.summary import write_summary
from rpbench.utils.io import load_jsonl

input_path = RUNS_DIR / 'demo_results.jsonl'
rows = load_jsonl(input_path)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = REPORT_DIR / 'summary_table.csv'
ols_fig_path = REPORT_DIR / 'ols_plot_eps_vs_mse.png'
covariance_fig_path = REPORT_DIR / 'covariance_plot_eps_vs_error.png'
md_path = REPORT_DIR / 'demo_summary.md'

build_summary_table(rows, csv_path)
ols_plot_eps_vs_mse(rows, ols_fig_path)
plot_eps_vs_covariance_error(rows, covariance_fig_path)
write_summary(rows, md_path, ols_fig_path.name, covariance_fig_path.name, csv_path.name)

print('Wrote:')
print(' -', csv_path)
print(' -', ols_fig_path)
print(' -', covariance_fig_path)
print(' -', md_path)


Wrote:
 - /home/wei402/Desktop/rp-benchmark/reports/summary_table.csv
 - /home/wei402/Desktop/rp-benchmark/reports/ols_plot_eps_vs_mse.png
 - /home/wei402/Desktop/rp-benchmark/reports/covariance_plot_eps_vs_error.png
 - /home/wei402/Desktop/rp-benchmark/reports/demo_summary.md


In [4]:
import pandas as pd

summary_csv = REPORT_DIR / 'summary_table.csv'
summary_md = REPORT_DIR / 'demo_summary.md'
ols_fig_png = REPORT_DIR / 'ols_plot_eps_vs_mse.png'
covariance_fig_png = REPORT_DIR / 'covariance_plot_eps_vs_error.png'

display(pd.read_csv(summary_csv))

print('\n--- demo_summary.md ---\n')
print(summary_md.read_text())

print('\nOLS figure path:', ols_fig_png)
print('Covariance figure path:', covariance_fig_png)


,mechanism,epsilon,test_mse_mean,test_mse_std,rel_fro_mean,rel_fro_std,runtime_mean,n_seeds
0,Blocki12_JL,0.5,7.795094,12.923770,401596.230423,35276.900411,0.001342,5
1,Blocki12_JL,1.0,7.794201,12.921951,100399.205515,8819.233919,0.001335,5
2,Blocki12_JL,2.0,7.790630,12.914682,25099.949288,2204.817297,0.001329,5
3,Blocki12_JL,4.0,7.776373,12.885663,6275.135231,551.213144,0.001329,5
4,Mech_RP,0.5,0.228677,0.127792,0.524595,0.100095,0.001309,5
5,Mech_RP,1.0,0.167570,0.063001,0.346763,0.072419,0.001287,5
6,Mech_RP,2.0,0.137186,0.025002,0.257851,0.064882,0.001284,5
7,Mech_RP,4.0,0.124314,0.010446,0.215318,0.063256,0.001284,5



--- demo_summary.md ---

# WF-001 Demo Run Summary

**Dataset:** autompg  
**Mechanisms:** Blocki12_JL, Mech_RP  
**Task:** OLSFromRelease  
**Epsilon grid:** [0.5, 1.0, 2.0, 4.0]  
**Delta:** 1.00e-06  
**Seed batch:** mode=fixed, base_seed=0, count=5  
**Expanded seeds:** [1874364848, 2613022947, 2968811710, 3141116543, 3964924996]  
**Trial indices:** [0, 1, 2, 3, 4]  
**Total records:** 41  

**Non-private baseline test MSE:** 0.109278

## Results

See [summary_table.csv](summary_table.csv) for the aggregate table.

### OLS Downstream Task

![OLS Downstream Utility Plot](ols_plot_eps_vs_mse.png)

### Covariance Release Quality

![Covariance Release Quality Plot](covariance_plot_eps_vs_error.png)

## Outputs

- Row-level results: `data/outputs/runs/demo_results.jsonl`
- Summary table: `summary_table.csv`
- OLS figure: `ols_plot_eps_vs_mse.png`
- Covariance figure: `covariance_plot_eps_vs_error.png`


OLS figure path: /home/wei402/Desktop/rp-benchmark/reports/ols_plot_eps_vs_mse.png